# TTM-Small-4M T4 gốc — chạy thật bước 1.025–1.280

**Đã được chủ dự án cho phép ngày 09/11/2026.** Chọn Runtime → Change runtime type → T4 GPU (Colab Free), sau đó Runtime → Run all. Đăng nhập/mount đúng Google Drive khi được hỏi.

Một lượt duy nhất với checkpoint cha SHA-pinned, hồi phục AdamW/scheduler/RNG/GradScaler, TRAIN/VAL gốc. Lưu READY bước 1152 và 1280 trên Drive. Không đọc TEST, không chạy EmbeddingGemma2, không sửa trainer/lineage/main. Nếu Colab ngắt, chỉ cho phép resume cùng RunID từ READY 1152; nếu checkpoint chưa tới 1152, dừng để phân xử, không chạy lại 1024.


In [ ]:
# TTM-Small-4M original T4 — authorized one-shot 1024→1280
# Owner authorization recorded 2026-11-09. Use Colab FREE with Tesla T4.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, subprocess, sys, urllib.request, shutil
import torch

RUN_ID = "ttm_small4m_step1024_to1280_exploratory_v1"
REPO = "minhtri22/TokenToModel-Lab"
SOURCE_COMMIT = "0b30376f4f4e377d277ba92ebb1dc82a07d2cf1c"
ORIGINAL_MAIN = "b57b44e7287f842e81050c1faee337f636333dd0"
ROOT = Path("/content/ttm_original_t4_1280_authorized")
DRIVE_BASE = Path("/content/drive/MyDrive/TTM/ttm-small-pilot")
PARENT = DRIVE_BASE/"t4_science_4m_1024_a2_20261108_v1/checkpoints/step_1024"
DATA = DRIVE_BASE/"t4_private_pack_20261108_v1/T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"
MANIFEST = DRIVE_BASE/"t4_private_pack_20261108_v1/T4_PRIVATE_PACK_MANIFEST.json"
OUTPUT = DRIVE_BASE/RUN_ID
FILES = {
  "ttm_t4_original_1024_to1280.py": (SOURCE_COMMIT,"tools/pilot_t4/original_1024to1280_20261109/ttm_t4_original_1024_to1280.py","1f539aedd04d7097f2cb26b0220b14bd106c8b65","55cf8f9f686d911411ed3284ba6baf00b7a4ed1d7cc255e3f4fed1b9a2f5365c"),
  "test_ttm_t4_original_1024_to1280_offline.py": (SOURCE_COMMIT,"tools/pilot_t4/original_1024to1280_20261109/test_ttm_t4_original_1024_to1280_offline.py","e10ce2f0ce78d1d9aa0a7a7170735fefe79a825c","fad6d4860493209a285276dc12f1fd26d47ceb694b4aa8e1e64edbde456f7e28"),
  "p02.py": (ORIGINAL_MAIN,"src/p02/transformer_core.py","9762343e06cf800b50574c2871ae1ed58537fb18",None),
  "t4_train_science_a2.py": (ORIGINAL_MAIN,"tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py","76c4d0155cd0736919f41682f240b47ad3d7c428",None),
  "t4_capacity_smoke.py": (ORIGINAL_MAIN,"tools/pilot_t4/integrated/t4_capacity_smoke.py","c9a6f256ba975228683ee7de2a1090e649110e97",None),
  "t4_eval.py": (ORIGINAL_MAIN,"tools/pilot_t4/integrated/t4_eval.py","bef006bdff173fba6d8ba4aa0ef687a1732fd7c4",None)
}
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()
if ROOT.exists():
    raise RuntimeError("LOCAL_BOOTSTRAP_ALREADY_EXISTS: restart in clean Colab runtime; do not delete Drive research")
ROOT.mkdir(parents=True)
for name,(commit,path,expected_blob,expected_sha) in FILES.items():
    url=f"https://raw.githubusercontent.com/{REPO}/{commit}/{path}"
    data=urllib.request.urlopen(url,timeout=60).read()
    blob=hashlib.sha1(b'blob '+str(len(data)).encode()+b'\0'+data).hexdigest()
    if blob!=expected_blob: raise RuntimeError(f"PINNED_GIT_BLOB_MISMATCH:{name}")
    if expected_sha and hashlib.sha256(data).hexdigest()!=expected_sha:
        raise RuntimeError(f"PINNED_SHA256_MISMATCH:{name}")
    (ROOT/name).write_bytes(data)
print("SOURCE_PINNED_PASS", flush=True)

# CPU negative QA; no research TRAIN occurs in this precheck.
qa=subprocess.run([sys.executable,str(ROOT/"test_ttm_t4_original_1024_to1280_offline.py")],cwd=ROOT,check=False)
if qa.returncode: raise RuntimeError("CPU_OFFLINE_QA_FAILED_NO_TRAIN")
print("CPU_OFFLINE_QA=PASS_22_OF_22", flush=True)

for path in (PARENT/"checkpoint.pt",PARENT/"READY.json",PARENT/"progress.json",DATA,MANIFEST):
    if not path.is_file() or path.is_symlink(): raise RuntimeError(f"MISSING_OR_LINKED_DRIVE_FILE:{path}")
if not torch.cuda.is_available() or torch.cuda.device_count()!=1 or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("COLAB_FREE_TESLA_T4_REQUIRED")
if str(torch.__version__)!="2.14.1+cu130" or str(torch.version.cuda)!="13.0":
    raise RuntimeError(f"PINNED_T4_TORCH_MISMATCH:{torch.__version__}:{torch.version.cuda}")
print("T4_GPU_ENVIRONMENT=PASS",torch.cuda.get_device_name(0),torch.__version__, flush=True)

common=[sys.executable,str(ROOT/"ttm_t4_original_1024_to1280.py"),
        "--parent-dir",str(PARENT),"--pack",str(DATA),"--pack-manifest",str(MANIFEST),
        "--p02",str(ROOT/"p02.py"),"--trainer",str(ROOT/"t4_train_science_a2.py"),
        "--capacity",str(ROOT/"t4_capacity_smoke.py"),"--evaluator",str(ROOT/"t4_eval.py"),
        "--output",str(OUTPUT)]
p=subprocess.run(common+["--mode","inspect"],check=False)
if p.returncode: raise RuntimeError("ORIGINAL_PARENT_INSPECT_FAILED_NO_TRAIN")

# This lock records the user's explicit owner approval, not a new request for approval.
lock={
  "authorization":"OWNER_APPROVED_ONESHOT_EXPLORATORY",
  "run_id":RUN_ID,
  "parent_checkpoint_sha256":"cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b",
  "source_sha256":sha(ROOT/"ttm_t4_original_1024_to1280.py"),
  "original_science_lock_sha256":"fd183ade2a696d40313aae19b63429d4ef3edd8abf25c899c8cc496836d36817",
  "train_val_pack_sha256":"184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913",
  "start_step":1024,"end_step":1280,"targets_per_step":4096,
  "precision":"fp16","microbatch_windows":8,
  "optimizer":"RESTORE_ORIGINAL_ADAMW","lr_after_1024":3e-5,
  "test_access":False,"q1q2_v3":False,"free_colab_only":True,
  "owner_approval_date":"2026-11-09",
  "purpose":"Only original T4 successor; A reference only; 1152 diagnostic, 1280 exploratory primary VAL"
}
lock_path=ROOT/"OWNER_SCIENCE_LOCK.json"
lock_path.write_text(json.dumps(lock,sort_keys=True,indent=2)+"\n",encoding="utf-8")
lock_sha=sha(lock_path)
print("OWNER_APPROVAL_LOCK_SHA256="+lock_sha,flush=True)
resume=[]
if OUTPUT.exists():
    if not OUTPUT.is_dir() or (OUTPUT/"RESULT_UNADJUDICATED.json").exists():
        raise RuntimeError("RUN_ALREADY_FINISHED_OR_INVALID_NO_RERUN")
    ready=OUTPUT/"checkpoints/step_1152/READY.json"
    if not ready.is_file(): raise RuntimeError("PARTIAL_RUN_WITHOUT_1152_READY_HOLD_NO_RESET")
    resume=["--resume-ready",str(ready.parent)]
    print("SAME_RUNID_RESUME_FROM_1152_ONLY",flush=True)
else:
    print("NEW_ONE_SHOT_RUN_1025_TO_1280",flush=True)

cmd=common+["--mode","execute","--lock",str(lock_path),"--lock-sha",lock_sha]+resume
rc=subprocess.run(cmd,check=False).returncode
if rc: raise RuntimeError(f"GPU_TRAIN_HOLD_EXIT_{rc}; inspect Drive evidence, do not create a new RunID")
if not (OUTPUT/"RESULT_UNADJUDICATED.json").is_file():
    raise RuntimeError("GPU_RUN_RETURNED_WITHOUT_RESULT")
for step in (1152,1280):
    folder=OUTPUT/"checkpoints"/f"step_{step:04d}"
    ready=json.loads((folder/"READY.json").read_text(encoding="utf-8"))
    if ready["step"]!=step or sha(folder/"checkpoint.pt")!=ready["sha256"] or sha(folder/"progress.json")!=ready["progress_sha256"]:
        raise RuntimeError(f"DRIVE_READY_READBACK_FAILED:{step}")
with (OUTPUT/"OWNER_SCIENCE_LOCK.json").open("x",encoding="utf-8") as f:
    f.write(lock_path.read_text(encoding="utf-8"))
result=json.loads((OUTPUT/"RESULT_UNADJUDICATED.json").read_text(encoding="utf-8"))
print("TTM_ORIGINAL1280_COMPLETED_UNADJUDICATED",flush=True)
print("VAL_NLL_1024",result["baseline_nll"],"VAL_NLL_1280",result["endpoint_nll"],flush=True)
print("DELTA",result["delta_nll"],"CI95",result["ci95"],"SIGNAL",result["verdict"],flush=True)
print("DRIVE_RUN_FOLDER",OUTPUT,flush=True)
